# Algorithm Design — KNN Variants for Class Imbalance

This notebook is the theoretical companion to the project. It derives every algorithm from first
principles, explains all design decisions, and documents open research directions.

**Reader:** this is written as a reference document, not a tutorial. Familiarity with basic
probability and KNN classification is assumed. Each section is self-contained.

**Contents**
1. Problem statement and structural failure mode analysis
2. KNNFairRank — full derivation from first principles
3. Development history — why v1 and v2 were wrong
4. The four structural failure modes of v3
5. Modification A — local imbalance ratio (dropped)
6. Modification B — magnitude-aware votes
7. Modification C — CV-tuned exponent α (headline variant)
8. Modification D — confidence-based per-query fallback (open)
9. Modification E — direct density-ratio formulation (open)
10. KNNAdaptiveTopo — persistent homology case classifier
11. Beyond FairRank: topology-aware local correction (open)

---
## 1. Problem Statement and Structural Failure Mode Analysis

### 1.1 The setting

We study binary classification with class imbalance. Training data consists of $N$ points
$(x_i, y_i)$ where $y_i \in \{0, 1\}$, with class counts $N_\text{maj}$ (majority, label 0)
and $N_\text{min}$ (minority, label 1), and imbalance ratio:

$$r = \frac{N_\text{maj}}{N_\text{min}} \gg 1$$

Standard KNN classifies a query point $x$ by a majority vote over its $k$ nearest training
neighbours. Under class imbalance this vote is systematically biased toward the majority class —
not because $x$ is geometrically close to the majority, but because majority points are simply
more numerous and therefore tend to dominate any neighbourhood.

### 1.2 Two levels of bias

**Surface bias (k-selection).** With large $k$ the neighbourhood swells and is overwhelmed by
majority points. The natural response is to shrink $k$ — and `KNNOptK` does this via inner
cross-validation, selecting the globally optimal $k$ for each dataset. This helps: inner CV
selects $k=1$ in ≈62% of folds in our benchmark, confirming that the optimal global response
is maximal locality. But performance still degrades with IR even at $k=1$, which points to a
deeper problem.

**Structural bias (rank comparison).** Even with $k=1$, standard KNN makes a fundamentally
unfair comparison: it asks whether the *closest minority* neighbour is closer than the *closest
majority* neighbour. With $N_\text{maj} \gg N_\text{min}$, majority points are much denser
in space, so their closest representative is statistically expected to be closer to any query
point — not because of geometry, but because of sampling density. Standard KNN conflates
geometric proximity with class density, and this conflation persists no matter how small $k$ is.

This is the core problem that KNNFairRank addresses.

### 1.3 Failure mode taxonomy

Once KNNFairRank (v3) corrects the structural bias under idealised assumptions, four residual
failure modes remain, each motivating a specific modification:

| ID | Name | Description |
|---|---|---|
| **F1** | Poisson-uniform violated locally | Minority points cluster rather than spread uniformly. Near a cluster the local density is high (correction over-compensates); far from any cluster it is meaningless. |
| **F2** | Global $r$ applied everywhere | The same $k_\text{eff} = r$ is used regardless of local class composition. Near majority-dense regions the correction should be stronger; near minority pockets it should be weaker. |
| **F3** | Binary votes discard margin | A comparison where $d^\text{min} = 1.00$ and $d^\text{maj} = 1.01$ contributes the same binary vote as one where $d^\text{min} = 0.1$ and $d^\text{maj} = 10$. |
| **F4** | No dampening mechanism | On datasets where v3 globally over-corrects, there is no dial to reduce the correction. |

Modifications A–E each target a subset of these failure modes.

---
## 2. KNNFairRank — Full Derivation from First Principles

### 2.1 The rank concept

When classifying query point $x$, sort all training points of each class by distance to $x$
separately. Define:

- $d_k^\text{min}(x)$ — distance from $x$ to the $k$-th nearest **minority** training point
- $d_k^\text{maj}(x)$ — distance from $x$ to the $k$-th nearest **majority** training point

"Rank $k$" simply means position $k$ in that sorted list. Rank 1 is the closest, rank 2 the
second closest, and so on. Standard KNN with $k=1$ implicitly compares rank-1 minority against
rank-1 majority and predicts the class whose rank-1 point is closer.

### 2.2 Statistical model of nearest-neighbour distances

Assume training points of class $c$ are drawn from a spatial distribution with local density
$\lambda_c$ (points per unit volume) in a $d$-dimensional feature space. Under a homogeneous
Poisson process approximation, the expected number of class-$c$ points inside a ball of radius
$\rho$ centred at $x$ is:

$$\mathbb{E}[\text{count inside ball}] = \lambda_c \cdot V_d \cdot \rho^d$$

where $V_d = \pi^{d/2} / \Gamma(d/2 + 1)$ is the volume of the unit $d$-ball. The
$k$-th nearest neighbour distance is the smallest $\rho$ such that the expected count reaches
$k$, giving:

$$\mathbb{E}[d_k^c(x)] \propto \left(\frac{k}{\lambda_c}\right)^{1/d}$$

This is the key scaling law. Two observations follow immediately:

1. **Density effect:** for fixed $k$, a denser class ($\lambda_c$ large) has smaller expected
   nearest-neighbour distances. Majority points are denser ($\lambda_\text{maj} > \lambda_\text{min}$)
   so $\mathbb{E}[d_1^\text{maj}] < \mathbb{E}[d_1^\text{min}]$ regardless of $x$.

2. **Rank effect:** increasing $k$ increases the expected distance as $k^{1/d}$.

### 2.3 Quantifying the unfair comparison

Under balanced class priors, $\lambda_\text{maj}/\lambda_\text{min} = N_\text{maj}/N_\text{min} = r$.
Comparing both classes at rank 1:

$$\frac{\mathbb{E}[d_1^\text{min}(x)]}{\mathbb{E}[d_1^\text{maj}(x)]}
= \frac{(1/\lambda_\text{min})^{1/d}}{(1/\lambda_\text{maj})^{1/d}}
= \left(\frac{\lambda_\text{maj}}{\lambda_\text{min}}\right)^{1/d}
= r^{1/d}$$

For $r = 10$ in $d = 2$ dimensions: $r^{1/d} = \sqrt{10} \approx 3.16$. The nearest minority
neighbour is expected to be **3× further away** than the nearest majority neighbour, purely due to
density. This bias grows as $r$ increases and shrinks (but does not vanish) as $d$ increases.

### 2.4 Deriving the fair majority rank

A fair comparison requires both sides to have the same expected distance under the null hypothesis
that $x$ lies on the decision boundary (where both classes are equally likely). We want the
majority rank $k_\text{eff}$ such that:

$$\mathbb{E}[d_1^\text{min}(x)] = \mathbb{E}[d_{k_\text{eff}}^\text{maj}(x)]$$

Substituting the scaling law:

$$\left(\frac{1}{\lambda_\text{min}}\right)^{1/d} = \left(\frac{k_\text{eff}}{\lambda_\text{maj}}\right)^{1/d}$$

Raising both sides to the power $d$ (eliminating the exponent):

$$\frac{1}{\lambda_\text{min}} = \frac{k_\text{eff}}{\lambda_\text{maj}}$$

$$\boxed{k_\text{eff} = \frac{\lambda_\text{maj}}{\lambda_\text{min}} = \frac{N_\text{maj}}{N_\text{min}} = r}$$

Note carefully: $k_\text{eff} = r$, **not** $r^{1/d}$. The distance ratio is $r^{1/d}$, but the
fair rank that corrects it is $r$. The $d$-th power cancels the $1/d$ exponent. This distinction
is crucial — confusing the two is exactly the error made in v1 and v2 (Section 3).

**Decision rule.** KNNFairRank predicts minority iff:

$$d_1^\text{min}(x) < d_r^\text{maj}(x)$$

i.e., the closest minority neighbour is closer than the $r$-th closest majority neighbour.

### 2.5 Multi-rank voting — reducing variance

A single comparison (rank-1 minority vs rank-$r$ majority) is noisy: it depends on whichever
single minority point and single majority point happen to be at those positions, which fluctuates
across query points. To reduce variance, KNNFairRank aggregates multiple independent fair
comparisons.

The same fairness argument scales to higher ranks: if rank-1 minority is fairly compared against
majority rank $r$, then rank-$i$ minority is fairly compared against majority rank $i \cdot r$:

$$\mathbb{E}[d_i^\text{min}(x)] = \left(\frac{i}{\lambda_\text{min}}\right)^{1/d}
= \left(\frac{i \cdot r}{\lambda_\text{maj}}\right)^{1/d}
= \mathbb{E}[d_{i \cdot r}^\text{maj}(x)]$$

For each vote $i = 1, 2, \ldots, n_\text{votes}$, cast:

$$\text{vote}_i =
\begin{cases}
+1 & \text{if } d_i^\text{min}(x) < d_{i \cdot r}^\text{maj}(x) \quad (\text{minority wins}) \\
-1 & \text{otherwise} \quad (\text{majority wins})
\end{cases}$$

Final prediction: minority iff $\sum_{i=1}^{n_\text{votes}} \text{vote}_i > 0$.

Each vote is an independent fair comparison by the same Poisson argument. Aggregating $n_\text{votes}$
comparisons dramatically reduces the sensitivity to any single lucky or unlucky point placement.
In our implementation $n_\text{votes} = 5$.

### 2.6 What the algorithm needs at inference time

For a query point $x$:
1. Compute distances to all training points
2. Sort minority distances → $d_1^\text{min} \leq d_2^\text{min} \leq \ldots$
3. Sort majority distances → $d_1^\text{maj} \leq d_2^\text{maj} \leq \ldots$
4. Compute $r = N_\text{maj}/N_\text{min}$ (stored from training)
5. For $i = 1, \ldots, n_\text{votes}$: compare $d_i^\text{min}$ vs $d_{\lfloor i \cdot r \rfloor}^\text{maj}$
6. Predict minority iff majority of votes favour minority

Note: no fitting is required beyond storing $N_\text{maj}$, $N_\text{min}$, and the training
points. KNNFairRank is a lazy learner like standard KNN.

---
## 3. Development History — Why v1 and v2 Were Wrong

Understanding the failures of v1 and v2 makes the v3 derivation more convincing.

### 3.1 The versions

| Version | $k_\text{eff}$ | Decision rule | Outcome |
|---|---|---|---|
| v1 | $r^{1/d}$ | Single binary comparison | Abandoned — under-corrects at high IR |
| v2 | $r^{1/d}$ | Multi-rank binary vote ($n_\text{votes}=5$) | Abandoned — same bug as v1 |
| **v3** | $r$ | Multi-rank binary vote | **Retained** — theoretically correct |

### 3.2 The v1/v2 mistake in detail

v1 and v2 used $k_\text{eff} = r^{1/d}$ instead of $k_\text{eff} = r$. Where does $r^{1/d}$
come from, and why is it wrong?

The distance ratio between the two rank-1 expected distances is:

$$\frac{\mathbb{E}[d_1^\text{min}]}{\mathbb{E}[d_1^\text{maj}]} = r^{1/d}$$

The v1/v2 reasoning was: *"the minority distance is $r^{1/d}$ times larger, so to compensate
we should advance the majority comparison rank by $r^{1/d}$."* This treats $k_\text{eff}$ as
a scale factor applied in distance space.

The error is conflating ranks and distances. Rank $k$ enters the distance scaling as $k^{1/d}$,
so to match a distance factor of $r^{1/d}$ you need:

$$k_\text{eff}^{1/d} = r^{1/d} \implies k_\text{eff} = r$$

v1/v2 stopped one step early — they computed the *distance ratio* $r^{1/d}$ and used it as the
*rank correction*, missing the $d$-th power that converts distance factors to rank factors.

### 3.3 Empirical consequence

At severe imbalance the difference is large. For $r = 100$, $d = 2$:
- v1/v2: $k_\text{eff} = 100^{1/2} = 10$ — a 10-fold rank offset
- v3: $k_\text{eff} = 100$ — a 100-fold rank offset

v1/v2 under-corrects by a factor of $r^{1-1/d}$. This factor grows with $r$ and shrinks toward
1 as $d \to \infty$. In high dimensions both versions converge (both $k_\text{eff} \to 1$),
but in the low-to-moderate dimensions typical of tabular data the gap is substantial.

### 3.4 From single comparison to multi-rank

The jump from v1 to v2 added multi-rank voting to v1's wrong $k_\text{eff}$. The motivation
was correct — single comparisons are noisy — but the bug in $k_\text{eff}$ remained. v3
combined the correct $k_\text{eff} = r$ derivation (Section 2) with multi-rank voting.

---
## 4. The Four Structural Failure Modes of v3

v3 is derived from a homogeneous Poisson process assumption: training points are distributed
uniformly and independently across the feature space. Real data violates this assumption in
several ways, each creating a specific failure mode.

### 4.1 F1 — Poisson-uniform violated locally

**What the assumption says.** Points are uniformly distributed at density $\lambda_c$ everywhere.

**What real data does.** Minority points cluster. Near a cluster, local minority density is high
($\lambda_\text{min}^\text{local} \gg \lambda_\text{min}^\text{global}$). Far from any
cluster, it is low. v3 uses the global ratio $r = N_\text{maj}/N_\text{min}$ everywhere, which
means:

- Near a minority cluster: $r$ over-estimates the true local ratio → over-corrects → flips too
  many majority queries to minority
- Far from clusters: $r$ under-estimates the true local difficulty → under-corrects

**Effect.** High dataset-to-dataset variance. On datasets with well-separated minority clusters
v3 over-corrects; on datasets with scattered minority points it under-corrects.

### 4.2 F2 — Global $r$ applied everywhere

**The problem.** Closely related to F1 but distinct: even if the minority class is uniformly
distributed, the majority class may not be. Near a majority-dense region, local $r_\text{local} \gg r$
and more correction is needed; near a majority-sparse region, less is needed. v3 uses the
same $k_\text{eff}$ everywhere.

**Effect.** Systematic errors at query points near majority density concentrations and minority
density concentrations.

### 4.3 F3 — Binary votes discard margin

**The problem.** The binary vote $\text{vote}_i \in \{+1, -1\}$ treats a comparison where
$d_i^\text{min} = 0.001$ and $d_{ir}^\text{maj} = 100$ identically to one where
$d_i^\text{min} = 0.49$ and $d_{ir}^\text{maj} = 0.51$. The former is very strong evidence
for minority; the latter is marginal. Aggregating binary votes discards this information.

**Effect.** The vote margin $\sum \text{vote}_i$ is a poor proxy for classification confidence.
Ambiguous query points (near the true decision boundary) and clear-cut points (deep inside a
class region) receive the same treatment.

### 4.4 F4 — No dampening mechanism

**The problem.** On a dataset where the Poisson-uniform assumption is severely violated globally
— for example, a highly clustered minority class in a low-dimensional space — v3 may
systematically over-correct everywhere. There is no parameter to reduce the correction.

**Effect.** On such datasets v3's G-mean can drop below SMOTE+KNN or even KNNOptK. The
algorithm has no fallback when its theoretical assumption is wrong.

---
## 5. Modification A — Local Imbalance Ratio (Dropped)

**Targets:** F1, F2

### 5.1 Idea

Replace the global ratio $r = N_\text{maj}/N_\text{min}$ with a per-query local estimate
$r_\text{local}(x)$. For each query point, look at a reference window of $K_\text{ref}$
nearest neighbours (from both classes combined) and compute the local ratio from the class
counts within that window:

$$r_\text{local}(x) = \frac{\text{count of majority in } K_\text{ref}\text{-window}}{\text{count of minority in } K_\text{ref}\text{-window}}$$

Then use $k_\text{eff}(x) = r_\text{local}(x)$ for this query.

**Intuition.** Near a minority cluster, the $K_\text{ref}$ window would contain many minority
points, giving $r_\text{local} < r$ and therefore a smaller correction. Near a sparse minority
region, $r_\text{local} > r$ and a stronger correction is applied. This adapts the correction
to the local class distribution as required by F1/F2.

### 5.2 Why it was dropped

The idea is appealing but collapses in the regime it is designed to help. When $N_\text{min}$
is small, the $K_\text{ref}$-window frequently contains **zero minority points**, making
$r_\text{local}$ undefined (division by zero). The only fix is to set a floor — but a floor
of 1 gives $r_\text{local} = K_\text{ref}$, which is an arbitrary large number unrelated to
the local geometry. A floor of $\epsilon > 0$ is similarly arbitrary.

The root cause is a fundamental sample-size restriction: to reliably estimate local class ratios
you need enough minority points in the window, but the whole problem is that minority points
are rare. The modification requires the very resource that imbalance denies.

**Formal condition for reliability.** For $r_\text{local}$ to have standard error less than
$\delta \cdot r_\text{local}$ (i.e. relative error below $\delta$), you need approximately:

$$n_\text{min}^\text{window} \geq \frac{1}{\delta^2 \cdot p_\text{min}^\text{window}}$$

where $p_\text{min}^\text{window}$ is the minority fraction in the window. At high IR,
$p_\text{min}^\text{window} \ll 1$, and the required count exceeds the total number of
minority points in the dataset. The modification is theoretically sound but practically infeasible
at the imbalance levels where it would matter most.

---
## 6. Modification B — Magnitude-Aware Votes

**Targets:** F3

### 6.1 The problem with binary votes

Standard v3 vote $i$:
$$\text{vote}_i = \begin{cases} +1 & d_i^\text{min} < d_{ir}^\text{maj} \\ -1 & \text{otherwise} \end{cases}$$

This is a hard threshold. The signed distance ratio $d_{ir}^\text{maj} / d_i^\text{min}$ — which
carries genuine information about how strongly minority-like or majority-like the query is —
is discarded.

### 6.2 Continuous score

Replace the binary vote with a soft score that preserves the distance ratio:

$$s_i = \frac{d_{ir}^\text{maj}(x)}{d_i^\text{min}(x) + d_{ir}^\text{maj}(x)} \in (0, 1)$$

**Interpretation:**
- $s_i = 0.5$ — the two distances are equal (exactly at the fair comparison boundary)
- $s_i \to 1$ — minority is much closer than majority at rank $ir$ (strong minority evidence)
- $s_i \to 0$ — majority is much closer (strong majority evidence)

The final decision: predict minority iff $\bar{s} = \frac{1}{n_\text{votes}} \sum_{i=1}^{n_\text{votes}} s_i > 0.5$.

This is equivalent to a thresholded mean of the soft votes. Unlike binary votes, a single
overwhelmingly clear comparison (large $s_i$) can outweigh several marginal comparisons in the
other direction.

### 6.3 Theoretical properties

The score $s_i$ can be interpreted as a probabilistic vote. If the $i$-th comparison distance
ratio $d_{ir}^\text{maj} / d_i^\text{min}$ follows a log-normal distribution under the Poisson
model, $s_i$ approximates the posterior probability $P(\text{minority} \mid \text{distances at rank } i)$.
This is not a rigorous derivation but motivates the form.

### 6.4 What it fixes and what it does not

**F3 (addressed).** The score preserves the margin of each comparison. A near-tie contributes
$s_i \approx 0.5$ (weak evidence) and a blowout contributes $s_i \approx 1$ (strong evidence).

**F1, F2 (not addressed).** The comparison ranks still use global $r$. The same
Poisson-uniform assumption applies.

**F4 (not addressed).** There is no mechanism to reduce the correction globally.

### 6.5 Empirical result

`KNNFairRankMagnitude` improves **ROC AUC** significantly (the soft scores give better
probability calibration than binary votes) but shows only marginal improvement on G-mean compared
to v3. This suggests that the binary vote's information loss (F3) matters more for ranking
quality than for hard-label classification quality.

---
## 7. Modification C — CV-Tuned Exponent α (Headline Variant)

**Targets:** F4

### 7.1 Motivation

v3 applies the correction $k_\text{eff} = r$ with no way to reduce it on datasets where the
Poisson-uniform assumption is violated. We need a tunable dial between:
- $\alpha = 0$: $k_\text{eff} = r^0 = 1$ — no correction (equivalent to standard KNN with $k=1$)
- $\alpha = 1$: $k_\text{eff} = r^1 = r$ — full v3 correction

For $\alpha \in (0, 1)$: a partial correction that discounts the theoretical value. The
parameter $\alpha$ modulates how strongly we trust the Poisson-uniform assumption.

### 7.2 The exponent parameterisation

Define the effective majority rank as:

$$k_\text{eff} = \text{round}(r^\alpha), \quad \alpha \in (0, 1]$$

The voting scheme is identical to v3: for $i = 1, \ldots, n_\text{votes}$, compare
$d_i^\text{min}$ against $d_{i \cdot k_\text{eff}}^\text{maj}$.

**Why this parameterisation?** The exponent $\alpha$ acts on the rank in the same space as
the original theoretical derivation. $k_\text{eff} = r^\alpha$ is the correction that would
be theoretically correct if the data's effective intrinsic dimensionality were $1/\alpha$
rather than 1 — i.e., it interpolates naturally between no correction and full correction in a
dimensionally-motivated way.

### 7.3 Selecting α by inner cross-validation

For each training fold, perform an inner $3$-fold stratified CV over the grid
$\alpha \in \{0.25, 0.5, 0.75, 1.0\}$, scored by G-mean (the primary metric). The $\alpha$
that maximises inner CV G-mean is used for prediction on the outer test fold.

This is the same pattern as `KNNOptK`'s inner CV for $k$ selection. The key difference:
`KNNOptK` selects a global $k$ for the whole dataset; Modification C selects $\alpha$ which
controls the *type* of correction, not just its scale.

**Grid choice.** The grid $\{0.25, 0.5, 0.75, 1.0\}$ spans from mild ($r^{0.25}$) to full
($r^1 = r$) correction in four steps. $\alpha = 0$ (no correction) is excluded because if the
data cannot benefit from any correction, `KNNOptK` already handles it.

### 7.4 v3 as a special case

When $\alpha = 1$, `KNNFairRankCV` reduces exactly to v3. When the Poisson-uniform assumption
holds well, inner CV should select $\alpha = 1$. The distribution of selected $\alpha$ values
across datasets is therefore diagnostic: if $\alpha = 1$ dominates, the assumption is
approximately valid; if lower values dominate, the assumption is frequently violated.

### 7.5 Addressing F1 and F2 indirectly

While $\alpha$ is still a global parameter per dataset (not per query), it partially addresses
F1 and F2 at the dataset level: if the minority class is highly clustered on a particular
dataset, the inner CV will tend to find that a smaller $\alpha$ (less aggressive correction)
produces better G-mean, effectively detecting and dampening the over-correction.

### 7.6 Empirical result

`KNNFairRankCV` achieves the **best average rank** across our benchmark, outperforming v3 in
rank stability while remaining competitive on mean G-mean. This confirms that the reliability
gap of v3 (F4) is real and that a data-driven $\alpha$ closes it.

---
## 8. Modification D — Confidence-Based Per-Query Fallback (Open Direction)

**Targets:** F4, partially F3

### 8.1 Idea

Modification C addresses F4 at the *dataset level* — it selects a global $\alpha$ that works
well on average across the dataset. But $\alpha$ may be correct on average yet wrong in specific
regions of the feature space (heterogeneous datasets). Modification D addresses F4 at the
*query-point level*.

When KNNFairRank's vote is close to the indifference point, the classifier is uncertain. In those
cases, defer to a reliable fallback (e.g. `KNNOptK`):

$$\hat{y}(x) = \begin{cases}
\hat{y}_\text{FairRank}(x) & \text{if } |\bar{s} - \tfrac{1}{2}| \geq \tau \\
\hat{y}_\text{fallback}(x) & \text{otherwise}
\end{cases}$$

where $\bar{s}$ is the mean vote score (from Modification B) or vote fraction (from v3), and
$\tau \in (0, \tfrac{1}{2})$ is a confidence threshold. When the score is close to 0.5
(ambiguous), the fallback takes over.

### 8.2 Implementation sketch

Wrap `KNNFairRank` in a thin meta-classifier that also holds a pre-fit `KNNOptK`. At predict
time: run FairRank, check the vote margin, and delegate per-query to the fallback if the margin
is below $\tau$.

### 8.3 Why it is lower priority after Modification C

D was originally motivated by the observation that FairRank's catastrophic failures concentrate
near the decision boundary — exactly the ambiguous region where $\bar{s} \approx 0.5$. But
once Modification C damps global over-correction via $\alpha$, most of those failures become
globally mitigated. D would then only add value on datasets where $\alpha$ is correct on
average but incorrect in specific pockets — a second-order effect.

**When D would still matter:**
- Datasets with strong spatial heterogeneity within a class (minority sub-groups with very
  different densities in different regions)
- Safety-critical settings where a hard worst-case guarantee relative to a baseline is required

### 8.4 Cost and risk

**Cost.** One extra hyperparameter ($\tau$), one extra model fit (the fallback), up to two
predict calls per query.

**Risk.** The algorithm becomes a conditional dispatcher, which breaks the "single principled
decision rule" property of v3. This makes it harder to analyse theoretically and harder to
explain.

---
## 9. Modification E — Direct Density-Ratio Formulation (Open Direction)

**Targets:** F1, F2

### 9.1 Idea

Rather than correcting the comparison rank as a proxy for density, estimate the class-conditional
local densities directly and apply Bayes' rule with balanced priors:

$$\hat{y}(x) = \text{minority} \iff \hat{p}(x \mid \text{min}) > \hat{p}(x \mid \text{maj})$$

The standard $k$-NN density estimator of class $c$ at query $x$ is:

$$\hat{p}(x \mid c) = \frac{k_c}{N_c \cdot V_d \cdot d_{k_c}^c(x)^d}$$

where $k_c$ is the number of class-$c$ neighbours used, $d_{k_c}^c(x)$ is the distance to the
$k_c$-th nearest class-$c$ neighbour, and $V_d$ is the volume of the unit $d$-ball. This
estimator captures local density variations: near a minority cluster, $d_{k_c}^\text{min}(x)$
is small and $\hat{p}(x \mid \text{min})$ is large.

### 9.2 v3 is a special case of E

Setting $k_\text{min} = 1$, $k_\text{maj} = k_\text{eff}$, and asking when
$\hat{p}(x \mid \text{min}) = \hat{p}(x \mid \text{maj})$ (the decision boundary):

$$\frac{1}{N_\text{min} \cdot d_1^\text{min}(x)^d} = \frac{k_\text{eff}}{N_\text{maj} \cdot d_{k_\text{eff}}^\text{maj}(x)^d}$$

At distance parity ($d_1^\text{min} = d_{k_\text{eff}}^\text{maj}$, the v3 decision
boundary) the distances cancel:

$$\frac{1}{N_\text{min}} = \frac{k_\text{eff}}{N_\text{maj}}
\implies k_\text{eff} = \frac{N_\text{maj}}{N_\text{min}} = r$$

**v3 is exactly Modification E with $k_\text{min} = 1$ and the distance-parity decision rule
under Poisson-uniform density.** Modification E is the general framework; v3 is the special
case where the local density estimate is replaced by the global class ratio.

### 9.3 What the generalisation buys

With $k_\text{min} > 1$, the density estimate for the minority class averages over more
neighbours, reducing noise. With $k_\text{maj}$ chosen jointly with $k_\text{min}$, the
optimal majority neighbourhood size adapts to local geometry rather than being forced to equal
$r \cdot k_\text{min}$.

More importantly, the full density estimate uses $d_{k_c}^c(x)^d$ rather than just comparing
two scalar distances. This captures local density variation (F1) without requiring a local ratio
estimate that collapses at small minority sample sizes (the failure of Modification A).

### 9.4 Implementation sketch

At predict time, for each query $x$:
1. Compute $d_{k_\text{min}}^\text{min}(x)$ and $d_{k_\text{maj}}^\text{maj}(x)$
2. Evaluate $\hat{p}(x \mid \text{min}) = k_\text{min} / (N_\text{min} \cdot d_{k_\text{min}}^{\text{min},d})$
3. Evaluate $\hat{p}(x \mid \text{maj}) = k_\text{maj} / (N_\text{maj} \cdot d_{k_\text{maj}}^{\text{maj},d})$
4. Predict minority iff $\hat{p}(x \mid \text{min}) > \hat{p}(x \mid \text{maj})$

The intrinsic dimensionality $d$ can be estimated locally using the Levina-Bickel MLE on the
$k$-NN distances, which is already computed in v3's internal `_estimate_lid` helper.

Hyperparameters: $k_\text{min}$ and $k_\text{maj}$ (or equivalently $k_\text{min}$ and
their ratio). These can be selected by inner CV — analogous to Modification C's $\alpha$.

---
## 10. KNNAdaptiveTopo — Persistent Homology Case Classifier

This was our first serious attempt at a locally adaptive algorithm before arriving at the
fair-rank insight. It represents the "local k adaptation" research direction.

### 10.1 Motivation

If we could characterise the local geometric structure around each query point, we could choose
$k$ accordingly:
- In a dense, homogeneous region: large $k$ is safe (the neighbourhood is clean)
- Near a minority cluster: small $k$ keeps us within the cluster
- On a boundary: moderate $k$ to smooth across it
- Near an outlier: large $k$ to look past the noise

The challenge is characterising local structure without labels (we don't know where the
boundary is at test time). **Persistent homology** provides a geometric description of structure
purely from point positions.

### 10.2 Persistent homology — the essentials

Given a set of points, build a filtration: a nested sequence of simplicial complexes
$K_0 \subseteq K_1 \subseteq \ldots$ parameterised by a distance threshold $\epsilon$.
At each $\epsilon$, connect pairs of points within distance $\epsilon$ by edges, triples
within distance $\epsilon$ by triangles, etc.

As $\epsilon$ grows, topological features (connected components, loops, voids) appear and
disappear. The **persistence** of a feature is the range of $\epsilon$ over which it exists.
Features with high persistence are robust structure; features with low persistence are noise.

The **Betti numbers** summarise the topology at a given $\epsilon$:
- $\beta_0$ (H0): number of connected components
- $\beta_1$ (H1): number of independent loops/holes

For our purposes, we use the total persistence of H0 and H1 features — a scalar summary of
how much topological structure exists in the local neighbourhood.

### 10.3 The four-case classifier

For each query point $x$, extract the $K_\text{ref}$ nearest training neighbours (combined
classes). Run persistent homology on this local point cloud and extract two statistics:

- $P_0$: total H0 persistence (normalised by diameter) — high if the neighbourhood has many
  persistent sub-clusters (fragmented structure)
- $P_1$: total H1 persistence (normalised by diameter) — high if the neighbourhood has
  significant loop structure (boundary-like)

Classify $x$ into one of four cases:

| Case | $P_0$ | $P_1$ | Interpretation | Action |
|---|---|---|---|---|
| Clean majority | Low | Low | Dense, connected, no loops → well inside majority region | $k = k_\text{max}$ |
| Minority cluster | Low minority fraction | Low | Neighbourhood is majority-dominated, query near minority edge | $k = k_\text{min}$ |
| Boundary | High | High | Complex structure with loops → near class boundary | $k = k_\text{mid}$ |
| Outlier/noise | High | Low | Many components but no loops → scattered points, outlier region | $k = k_\text{max}$ |

The thresholds on $P_0$ and $P_1$ (and $k_\text{min}, k_\text{mid}, k_\text{max}$) are
set in `config/settings.yaml`.

### 10.4 Why it helps over KNNOptK

`KNNOptK` selects a single $k$ by cross-validation on the whole dataset — it is globally optimal
but locally wrong. KNNAdaptiveTopo selects $k$ per-query based on what the local geometry
actually looks like. On the benchmark it achieves G-mean ≈ 0.618 vs KNNOptK's 0.600 — a
meaningful improvement confirming that local adaptation is the right direction.

### 10.5 Why the fair-rank family is better

KNNAdaptiveTopo still uses the standard KNN decision rule once $k$ is selected — it fixes
*how many* neighbours to look at but not *how to compare* the two classes. The fair-rank family
fixes the comparison itself. Empirically, `KNNFairRank` (G-mean 0.787) substantially outperforms
`KNNAdaptiveTopo` (0.618), which confirms the structural insight from Section 1: adapting $k$
is the wrong frame for the problem.

### 10.6 Limitations

1. **Computational cost.** Persistent homology on a local neighbourhood of $K_\text{ref}$
   points costs $O(K_\text{ref}^3)$ in the worst case for the 2-skeleton. At inference time
   this runs per query point, making the algorithm significantly slower than FairRank.

2. **Curse of dimensionality.** In high dimensions ($d \gtrsim 20$), pairwise distances
   concentrate — all points become approximately equidistant. Persistence diagrams flatten out:
   every feature has approximately equal birth and death time. The four-case classifier loses
   discriminative power. This is why `hiva_agnostic` ($d = 1617$) is excluded from our
   benchmark.

3. **Noisy minority topology.** With few minority points ($N_\text{min} \ll N_\text{maj}$),
   the local minority topology is unreliable — apparent clusters may be sampling gaps, apparent
   loops may be missing minority points. This connects to the circular problem explored in
   Section 11.

---
## 11. Beyond FairRank: Topology-Aware Local Correction (Open Direction)

### 11.1 The remaining limitation of FairRank

FairRank applies the same correction $k_\text{eff} = r$ (or $r^\alpha$ in Modification C)
to every query point. This is the consequence of the Poisson-uniform assumption: global class
counts are sufficient statistics for the correction. The full density-ratio formulation
(Modification E) relaxes this to:

$$k_\text{eff}(x) = \frac{\hat{\lambda}_\text{maj}(x)}{\hat{\lambda}_\text{min}(x)}$$

where $\hat{\lambda}_c(x)$ is the local spatial density of class $c$ near $x$. This is a
strictly better correction: it reduces to FairRank when density is uniform, and adapts when it
is not. The natural tool for estimating local density structure without parametric assumptions
is **persistent homology**.

### 11.2 The circular problem

Using persistent homology to estimate local minority density creates a circular dependency:

1. Persistent homology needs enough points to reliably detect structure
2. With $N_\text{maj}$ majority samples the topology is well-defined: clusters are dense,
   boundaries are sharp, persistence diagrams are stable
3. With $N_\text{min} \ll N_\text{maj}$ minority samples the topology is noisy: features
   appear and disappear with small perturbations, cluster boundaries are vague, apparent holes
   may be sampling gaps rather than true voids

If you estimate $\hat{\lambda}_\text{min}(x)$ from a noisy minority persistence diagram,
you have not escaped the sampling bias — you have hidden it inside a less transparent estimation
step. The majority topology is trusted more than minority topology by construction, and the
resulting local ratio is pulled toward majority, which is precisely the artifact FairRank
was designed to remove.

**The circular problem:** to use topology for density estimation you need reliable topology,
but reliable minority topology requires sample sizes that class imbalance denies.

### 11.3 Breaking the circularity: topological certainty from sample size

The circularity can be broken by quantifying topological uncertainty as a function of sample
size — and then weighting estimates by their confidence.

**Niyogi-Smale-Weinberger (2008) bound.** Given a compact $d$-dimensional submanifold
$\mathcal{M}$ with reach $\tau$ (a geometric measure of curvature and thickness), $N$ points
drawn uniformly at random satisfy:

$$N \geq C \cdot \frac{1}{\tau^d} \log \frac{1}{\delta}
\implies P\bigl(\text{homology}(\hat{\mathcal{M}}) = \text{homology}(\mathcal{M})\bigr) \geq 1 - \delta$$

Inverting this: given $N_\text{min}$ minority samples and a prior on $\tau$ (geometric
complexity), we can compute a confidence $p_\text{topo}(N_\text{min}, \tau, d) \in [0, 1]$
that the detected minority topology is the true topology.

**Bootstrap confidence bands (Chazal et al. 2017).** An alternative that does not require
knowledge of $\tau$: repeatedly subsample the minority point cloud, compute persistence
diagrams on each subsample, and measure the stability of each topological feature across
subsamples. Features that survive subsampling are real structure; features that vanish are
sampling artifacts. This assigns per-feature confidence scores without geometric priors.

### 11.4 The proposed algorithm

Compute separate majority and minority persistence diagrams. Weight each topological density
estimate by its confidence:

$$\hat{\lambda}_c^\text{weighted}(x) = p_\text{topo}(N_c) \cdot \hat{\lambda}_c^\text{topo}(x) + (1 - p_\text{topo}(N_c)) \cdot \hat{\lambda}_c^\text{global}(x)$$

where $\hat{\lambda}_c^\text{global}(x) = N_c / V$ is the global (FairRank-style) density
estimate. When $N_c$ is large, the topological estimate is trusted; when $N_c$ is small (as
with the minority class under severe imbalance), we fall back toward the global estimate —
which is precisely what FairRank uses. The algorithm **interpolates between FairRank and
full topology-aware estimation** based on the statistical reliability of the minority topology.

### 11.5 Why this is promising

- When $N_\text{min}$ is large enough for reliable topology, the algorithm captures local
  density variation that FairRank misses (addressing F1 and F2)
- When $N_\text{min}$ is small (high imbalance), the algorithm gracefully degrades to
  FairRank — it does not become *worse* than the baseline it extends
- The confidence weighting is principled and computable, not a heuristic

### 11.6 Open questions

- What geometric prior on $\tau$ is appropriate for tabular data?
- How to estimate $\tau$ from the data without assuming the manifold structure?
- Does the bootstrap band approach work reliably when $N_\text{min} < 50$?
- How does the algorithm perform in the moderate-IR regime (r ≈ 5–15) where
  neither FairRank nor topology alone is clearly superior?